# 8. Final Model Testing

In [2]:
from inspect import signature
from chess_eval import *

## 8.1 Comparison

We think the _winner_ from the previous notebook is clear here: ExtremeGradientBoosting achieves the best results, as high as 0.64 in a fraction of the time that the KNN and Ridge models take. We can choose the best parameters and run a quick test to validate this claim. For this final test, we will use the random one million rows dataset we have stored as `one_full_rd` to make the test even more significant and robust.

In [4]:
dm = load_dataset("one_full_rd")
results = {}
full_results = {}

We can reuse the code from the sixth notebook adding the stored optimal parameters.

In [5]:
with tqdm(OPTIMAL_MODELS.items(), desc="Comparing models", leave=False) as pbar:  # ~10 minutes
    for name, cls in pbar:
        pbar.set_postfix({"model": name})
        if "random_state" in signature(cls).parameters:
            mm = ModelManager(cls(random_state=RANDOM_STATE, **OPTIMAL_PARAMS[name]))
        else:
            mm = ModelManager(cls(**OPTIMAL_PARAMS[name]))

        all_results, summary = mm.cross_validate(dm)

        full_results[name] = all_results

        results[name] = {
            "Fit time": summary["train_time_mean"][0],
            "Prediction time": summary["test_time_mean"][0],
            "Spearman Rank": summary["test_score_mean"][0],
        }

Present the results in a nice dataframe.

In [6]:
pd.DataFrame(results).transpose().sort_values(by="Spearman Rank", ascending=False)

,Fit time,Prediction time,Spearman Rank
XGBRegressor,201.663251,6.351394,0.644237
Ridge,1.494497,0.179520,0.519979
KNeighborsRegressor,0.488758,54.966543,0.372962


We get very decisive results. The decision-based model XGBRegressor is the best-performing by a big margin. It maintains the high score obtained in the previous notebook even if the training dataframe is now bigger, and even more shuffled then before (because of the structure of the `csv` file mentioned in the first notebook). The linear model Ridge keeps the score, but it is significantly lower to the XBGR model. The KNN model fails terribly for this dataframe. Since it relies on similar or correlated datapoints, and the dataframe used here is fairly random, for this size of the training data, there just isn't enough representation for all positions.

While it is true that the XGBR model is the one that takes the longest to train (the linear is basically instantaneous and KNN only takes time when predicting), we think the trade-off is justified by the score improvement.

Further research and testing using an actual chess engine with time limitations have to be done to definitely determine the best model, but for now we will choose the XGB regressor.

## 8.1 XGBRegressor Testing